<a href="https://colab.research.google.com/github/Souparnoo/Google_Collab_Projects/blob/main/notebook1_numpy_cv_crashcourse.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 1 — Python & NumPy crash course for Computer Vision
**Goal:** Understand that an image is just a NumPy array of numbers.
Run every cell top to bottom. Read every comment — they're the lesson.

---

## Part 1 — Python basics you must know

In [1]:
# ── Lists ──────────────────────────────────────────────────────────────
# A list holds multiple values. Square brackets, comma-separated.
pixel = [255, 128, 0]        # one pixel: Blue=255, Green=128, Red=0
print('pixel:', pixel)
print('blue channel:', pixel[0])   # index 0 = first item
print('red channel: ', pixel[2])   # index 2 = third item

# Lists can hold lists (this is how a row of pixels looks)
row = [[255,0,0], [0,255,0], [0,0,255]]  # 3 pixels: blue, green, red
print('row:', row)
print('middle pixel:', row[1])

pixel: [255, 128, 0]
blue channel: 255
red channel:  0
row: [[255, 0, 0], [0, 255, 0], [0, 0, 255]]
middle pixel: [0, 255, 0]


In [ ]:
# ── For loops ──────────────────────────────────────────────────────────
# You'll use loops constantly to process batches of images
image_files = ['dog.jpg', 'cat.jpg', 'car.jpg']

for filename in image_files:
    print(f'Processing: {filename}')   # f-string: {} inserts a variable

# Loop with enumerate() gives you index AND value — very useful
for i, filename in enumerate(image_files):
    print(f'  [{i}] {filename}')

In [ ]:
# ── Functions ──────────────────────────────────────────────────────────
# Functions let you reuse code. def = define, return = output.
def resize_dimensions(width, height, target=640):
    """Calculate new dimensions keeping aspect ratio."""
    scale = target / max(width, height)
    new_w = int(width  * scale)
    new_h = int(height * scale)
    return new_w, new_h

w, h = resize_dimensions(1920, 1080)
print(f'1920x1080 → {w}x{h}')   # should be 640x360

w, h = resize_dimensions(480, 640)
print(f'480x640  → {w}x{h}')    # should be 480x640

In [ ]:
# ── Conditionals ───────────────────────────────────────────────────────
# You'll use these for checking if images loaded correctly, etc.
def check_image(img):
    if img is None:
        return 'ERROR: image did not load'
    elif img.shape[0] < 32 or img.shape[1] < 32:
        return 'WARNING: image is very small'
    else:
        return f'OK: {img.shape[1]}x{img.shape[0]} image'

# We'll test this properly once we load real images below
print(check_image(None))   # → ERROR

## Part 2 — NumPy: the engine behind every image

In [ ]:
import numpy as np

# ── What is a NumPy array? ─────────────────────────────────────────────
# NumPy arrays are like Python lists BUT:
#   - Much faster (C under the hood)
#   - Every element is the same type
#   - Support math operations on the whole array at once

# 1D array — like a single row of pixel values
row = np.array([10, 20, 30, 40, 50], dtype=np.uint8)
print('1D array:', row)
print('shape:', row.shape)   # (5,) — 5 elements
print('dtype:', row.dtype)   # uint8 = unsigned 8-bit integer (0-255)

# Math on the whole array at once — no loop needed!
print('doubled:', row * 2)
print('halved: ', row // 2)

In [ ]:
# ── 2D array = grayscale image ─────────────────────────────────────────
# A grayscale image is a 2D grid of pixel brightness values (0=black, 255=white)

# Create a 6x6 grayscale image manually
gray_img = np.array([
    [  0,   0,   0,   0,   0,   0],   # black row
    [  0, 255, 255, 255, 255,   0],
    [  0, 255,   0,   0, 255,   0],   # hollow square
    [  0, 255,   0,   0, 255,   0],
    [  0, 255, 255, 255, 255,   0],
    [  0,   0,   0,   0,   0,   0],   # black row
], dtype=np.uint8)

print('shape:', gray_img.shape)  # (6, 6) — rows, cols
print('pixel at [2,2]:', gray_img[2, 2])   # 0 = black (inside the square)
print('pixel at [1,1]:', gray_img[1, 1])   # 255 = white (border)

In [ ]:
# ── 3D array = colour image ────────────────────────────────────────────
# Shape is (height, width, channels)
# OpenCV uses BGR order: channel 0=Blue, 1=Green, 2=Red

# Create a tiny 4x4 colour image
colour_img = np.zeros((4, 4, 3), dtype=np.uint8)   # all black

# Paint each quadrant a different colour (BGR values)
colour_img[0:2, 0:2] = [255,   0,   0]   # top-left     → blue
colour_img[0:2, 2:4] = [  0, 255,   0]   # top-right    → green
colour_img[2:4, 0:2] = [  0,   0, 255]   # bottom-left  → red
colour_img[2:4, 2:4] = [255, 255,   0]   # bottom-right → cyan

print('shape:', colour_img.shape)   # (4, 4, 3)
print('top-left pixel:', colour_img[0, 0])    # [255,   0,   0]
print('top-right pixel:', colour_img[0, 3])   # [  0, 255,   0]

# Total number of values stored
print('total values:', colour_img.size)   # 4*4*3 = 48
# A real 640x640 image = 640*640*3 = 1,228,800 values!

## Part 3 — Array slicing (the most important skill in CV)

In [ ]:
# ── Slicing syntax: array[start:stop] ─────────────────────────────────
# stop is EXCLUSIVE — array[0:3] gives indices 0, 1, 2
# This is how you CROP images

# Make a 10x10 image filled with the pixel's column number (easy to track)
img = np.zeros((10, 10), dtype=np.uint8)
for col in range(10):
    img[:, col] = col * 25   # each column has a different brightness

print('full image (first row):', img[0])

# Crop: rows 2-5, columns 3-7
crop = img[2:6, 3:8]
print('crop shape:', crop.shape)   # (4, 5) — 4 rows, 5 cols
print('crop first row:', crop[0])

In [ ]:
# ── Slicing a real-world crop ──────────────────────────────────────────
# Imagine a 480x640 image. You want just the top-right 200x200 corner.
#
#   img[row_start : row_end,  col_start : col_end]
#       ↑ y-axis              ↑ x-axis
#
# Remember: rows = height (y), cols = width (x)

fake_image = np.zeros((480, 640, 3), dtype=np.uint8)

# Crop top-right 200x200
top_right = fake_image[0:200, 440:640]
print('top-right crop shape:', top_right.shape)   # (200, 200, 3)

# Extract just the Blue channel
blue_channel = fake_image[:, :, 0]   # : means 'all'
print('blue channel shape:', blue_channel.shape)  # (480, 640)

# Extract a single row of pixels
middle_row = fake_image[240, :, :]   # row 240 (vertical midpoint)
print('middle row shape:', middle_row.shape)      # (640, 3)

## Part 4 — Key NumPy operations for image processing

In [ ]:
# ── Creating arrays ────────────────────────────────────────────────────
black  = np.zeros((100, 100, 3), dtype=np.uint8)   # all 0s  → black
white  = np.full((100, 100, 3), 255, dtype=np.uint8) # all 255 → white
random = np.random.randint(0, 256, (100, 100, 3), dtype=np.uint8)  # noise

print('black  mean:', black.mean())    # 0.0
print('white  mean:', white.mean())    # 255.0
print('random mean:', random.mean().round(1))  # ~127.5

In [ ]:
# ── dtype and why it matters ───────────────────────────────────────────
# uint8:   integers 0–255         ← storage, display
# float32: decimals 0.0–1.0       ← model input
# int32:   signed integers        ← intermediate calculations

img_uint8 = np.array([200, 100, 50], dtype=np.uint8)

# DANGER: uint8 overflow (wraps around!)
print('200 + 100 in uint8:', np.uint8(200) + np.uint8(100))   # 44, not 300!

# SAFE: convert to float first, then do math
img_float = img_uint8.astype(np.float32)
result = img_float + 100
print('200 + 100 in float32:', result[0])   # 300.0 ✓

# Clip back to valid range before saving
clipped = np.clip(result, 0, 255).astype(np.uint8)
print('clipped:', clipped)   # [255, 200, 150]

In [ ]:
# ── Normalisation ──────────────────────────────────────────────────────
# Neural networks expect pixel values as 0.0–1.0, not 0–255
# This is called normalisation

img = np.array([[[255, 128, 0], [64, 200, 32]]], dtype=np.uint8)
print('original range:', img.min(), '–', img.max())   # 0 – 255

img_norm = img.astype(np.float32) / 255.0
print('normalised range:', img_norm.min().round(3), '–', img_norm.max())  # 0.0 – 1.0
print('sample pixel:', img_norm[0, 0])   # [1.0, 0.502, 0.0]

# To go back to uint8
img_back = (img_norm * 255).astype(np.uint8)
print('restored:', img_back[0, 0])   # [255, 128, 0]

In [ ]:
# ── Reshape and transpose ──────────────────────────────────────────────
# Different frameworks expect different axis orders:
#   OpenCV / PIL:   (height, width, channels)  — HWC
#   PyTorch models: (channels, height, width)  — CHW

img_hwc = np.zeros((224, 224, 3), dtype=np.float32)   # HWC
print('HWC shape:', img_hwc.shape)   # (224, 224, 3)

# Swap to CHW for PyTorch
img_chw = img_hwc.transpose(2, 0, 1)   # move axis 2 → axis 0
print('CHW shape:', img_chw.shape)   # (3, 224, 224)

# Add batch dimension: (batch, channels, height, width)
img_batch = img_chw[np.newaxis, :]   # or np.expand_dims(img_chw, 0)
print('batch shape:', img_batch.shape)   # (1, 3, 224, 224)

## Part 5 — Putting it all together with a real image

In [ ]:
import cv2
import requests
from google.colab.patches import cv2_imshow
import matplotlib.pyplot as plt

# Download a reliable test image
url = 'https://upload.wikimedia.org/wikipedia/commons/a/a7/Camponotus_flavomarginatus_ant.jpg'
r = requests.get(url)
arr = np.frombuffer(r.content, np.uint8)
img = cv2.imdecode(arr, cv2.IMREAD_COLOR)

print('=== Image as a NumPy array ===')
print(f'Type:    {type(img)}')
print(f'Shape:   {img.shape}  ← (height, width, channels)')
print(f'Dtype:   {img.dtype}  ← uint8 = values 0-255')
print(f'Min:     {img.min()}')
print(f'Max:     {img.max()}')
print(f'Mean:    {img.mean():.1f}')
print(f'Size:    {img.size:,} values total ({img.shape[0]}×{img.shape[1]}×3)')

cv2_imshow(img)

In [ ]:
# ── Inspect individual pixels ──────────────────────────────────────────
h, w = img.shape[:2]

# Sample pixels from four corners
corners = {
    'top-left':     img[0,   0   ],
    'top-right':    img[0,   w-1 ],
    'bottom-left':  img[h-1, 0   ],
    'bottom-right': img[h-1, w-1 ],
    'centre':       img[h//2, w//2],
}
for name, pixel in corners.items():
    b, g, r = pixel
    print(f'{name:15s} B={b:3d} G={g:3d} R={r:3d}')

In [ ]:
# ── Visualise the three colour channels separately ─────────────────────
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
titles = ['Original (BGR→RGB)', 'Blue channel', 'Green channel', 'Red channel']

# Original — convert BGR→RGB for correct display in matplotlib
axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
axes[0].set_title(titles[0])

# Each colour channel shown as grayscale (brightness = that colour's intensity)
for i, (ax, title) in enumerate(zip(axes[1:], titles[1:])):
    ax.imshow(img[:, :, i], cmap='gray', vmin=0, vmax=255)
    ax.set_title(title)

for ax in axes:
    ax.axis('off')

plt.tight_layout()
plt.show()

print('Brighter = more of that colour in that region')

In [ ]:
# ── Crop using array slicing ───────────────────────────────────────────
h, w = img.shape[:2]

# Crop the centre 50% of the image
y1, y2 = h // 4, 3 * h // 4
x1, x2 = w // 4, 3 * w // 4

crop = img[y1:y2, x1:x2]   # pure NumPy slicing — no OpenCV needed!
print(f'Original: {img.shape} → Crop: {crop.shape}')
cv2_imshow(crop)

In [ ]:
# ── Full preprocessing pipeline ───────────────────────────────────────
# This is what you'll run on every image before feeding to a model

def preprocess(img, target_size=640):
    """Takes a BGR uint8 image, returns a float32 RGB array ready for a model."""
    # 1. Resize to square
    resized = cv2.resize(img, (target_size, target_size))
    # 2. Convert BGR → RGB (OpenCV loads as BGR, models expect RGB)
    rgb = cv2.cvtColor(resized, cv2.COLOR_BGR2RGB)
    # 3. Normalise to 0.0–1.0
    normalised = rgb.astype(np.float32) / 255.0
    return normalised

ready = preprocess(img)
print(f'Input:  shape={img.shape},   dtype={img.dtype},   range=[{img.min()}, {img.max()}]')
print(f'Output: shape={ready.shape}, dtype={ready.dtype}, range=[{ready.min():.1f}, {ready.max():.1f}]')
print('\nModel-ready!')

## Summary — what you learned

| Concept | Key takeaway |
|---|---|
| Image = NumPy array | Shape is always `(height, width, channels)` |
| dtype uint8 | Values 0–255. Beware overflow — convert to float32 before math |
| Slicing | `img[y1:y2, x1:x2]` crops an image. No OpenCV needed. |
| BGR vs RGB | OpenCV = BGR. Matplotlib/models = RGB. Always convert. |
| Normalise | Divide by 255.0 before feeding to any neural network |
| Channels | `img[:,:,0]` = Blue, `img[:,:,1]` = Green, `img[:,:,2]` = Red |

**Next:** Notebook 2 — Core image operations with OpenCV